# Video-frame adaptation ? separate YOLOv8 Colab run

This notebook starts from the completed **baseline `models/best.pt`**, combines the original 498 training images with reviewed images extracted from `1000030242.mp4`, and preserves the original 116 validation / 73 test images. The baseline model, metrics, data and earlier video remain unchanged.

**The source video is now training material.** Improvements on that same video are training-scene checks, not independent video accuracy. Best-checkpoint selection still uses the original validation split. No random split of adjacent video frames is made.

The requested baseline settings are kept: CUDA device 0, batch 16, 100 epochs, imgsz 640, workers 4, seed 42. Batch falls to 8 or 4 only after an actual CUDA out-of-memory error. The paused Windows CPU training is unrelated and must stay paused.


## 1. Configure a new, isolated run
The reviewed ZIP must be at `MyDrive/yolov8-stationery-tracking/video_finetune/video_training_addition_v2.zip`. Original dataset parts stay in their existing Drive project folder. Run this notebook once from top to bottom. A new run ID never resets a previous checkout.


In [ ]:
from pathlib import Path
import datetime
import json
import os
import re
import shutil
import subprocess
import sys
import uuid

if not Path('/content').is_dir():
    raise RuntimeError('Run this notebook in Google Colab; local CPU fallback is disabled.')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
REPOSITORY_URL = 'https://github.com/jeongjunseo/yolov8-stationery-tracking.git'
REPO_REF = 'main'
REPO = Path('/content') / ('stationery_video_finetune_' + RUN_ID)
DRIVE_PROJECT = Path('/content/drive/MyDrive/yolov8-stationery-tracking')
DRIVE_ADDITION = DRIVE_PROJECT / 'video_finetune/video_training_addition_v2.zip'
ADDITION_SHA256 = '22a7de84e451f3ad9c120773b74f42111c3955864ed21013d7614e83a980f4c9'
BASELINE_SHA256 = '72eec58f57a0decd694a98606e9759824cb7cb4d75af345e9ae8db735a4d66ea'
DATA_ARCHIVE = Path('/content/stationery_dataset.zip')
BACKUP_RUN = DRIVE_PROJECT / 'video_finetune/colab_runs' / RUN_ID
ADDITION_ROOT = REPO / 'runs/video_finetune_data/addition'
MIXED_DATA = REPO / 'runs/video_finetune_data/mixed.yaml'
print('Separate video-adaptation run ID:', RUN_ID)


## 2. Check GPU and mount Drive
Select a GPU runtime. The notebook refuses CPU training. Complete Google Drive authorization if Colab asks.


In [ ]:
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-c',
    "import torch; assert torch.cuda.is_available(), 'CUDA GPU is required'; "
    "print('PyTorch:', torch.__version__); print('GPU:', torch.cuda.get_device_name(0))"], check=True)
from google.colab import drive
drive.mount('/content/drive')
if not DRIVE_ADDITION.is_file():
    raise FileNotFoundError(f'The reviewed video-frame ZIP is missing: {DRIVE_ADDITION}')
if not re.fullmatch('[0-9a-f]{64}', ADDITION_SHA256):
    raise RuntimeError('Publish the reviewed addition ZIP SHA-256 before running this notebook.')
BACKUP_RUN.mkdir(parents=True, exist_ok=False)
print('Every-epoch backups and final ZIP:', BACKUP_RUN)


## 3. Fresh checkout and pinned training dependencies
Colab CUDA PyTorch/torchvision versions are retained. No earlier checkout is pulled over, reset, or deleted.

After the new clone succeeds, previously published adaptation artifacts are preserved in a new sibling `_published_snapshot` directory with a file/hash manifest. Only the reviewed annotation thumbnail is copied back as reusable input. Baseline files remain unchanged. Existing or active checkouts are rejected; no prior training result is deleted or reused as a new result.


In [ ]:
if REPO.exists():
    raise FileExistsError(f'Preserving existing checkout: {REPO}')
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPOSITORY_URL, str(REPO)], check=True)
# This runs only immediately after the guarded fresh clone, before sessions/data.
snapshot_result = subprocess.run(
    [sys.executable, str(REPO / 'scripts/preserve_published_finetune.py'), str(REPO)],
    check=True, capture_output=True, text=True, encoding='utf-8')
PUBLISHED_SNAPSHOT = json.loads(snapshot_result.stdout)
print('Previously published artifacts preserved:', PUBLISHED_SNAPSHOT['snapshot_directory'])
for required in ('scripts/finetune_video.py', 'scripts/check_dataset.py', 'models/best.pt'):
    if not (REPO / required).is_file():
        raise FileNotFoundError(f'Publish the adaptation code before running: {required}')
SESSION = REPO / 'runs/video_finetune_sessions' / RUN_ID
SESSION.mkdir(parents=True, exist_ok=False)
OUT = REPO / 'outputs/video_finetune'
if OUT.exists():
    raise FileExistsError('This checkout already contains adaptation outputs; preserve that experiment.')
OUT.mkdir(parents=True)
(OUT / 'published_snapshot.json').write_text(json.dumps(PUBLISHED_SNAPSHOT, indent=2) + '\n', encoding='utf-8')
from importlib.metadata import version
constraints = SESSION / 'constraints.txt'
constraints.write_text('torch==' + version('torch') + '\n' + 'torchvision==' + version('torchvision') + '\n', encoding='utf-8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements.txt'),
                'ultralytics==8.4.215', '-c', str(constraints), '--disable-pip-version-check'], check=True)
pip_check = subprocess.run([sys.executable, '-m', 'pip', 'check'], text=True, capture_output=True)
print(pip_check.stdout, pip_check.stderr)
(OUT / 'pip_check.json').write_text(json.dumps({'exit_code': pip_check.returncode,
    'stdout': pip_check.stdout, 'stderr': pip_check.stderr}, indent=2) + '\n', encoding='utf-8')
environment_code = """
import json, platform, subprocess
from pathlib import Path
from importlib.metadata import version
import torch, ultralytics, cv2
assert torch.cuda.is_available(), 'CUDA became unavailable after dependency installation'
result = {'runtime': 'Google Colab', 'python': platform.python_version(),
    'packages': {n: version(n) for n in ('torch','torchvision','ultralytics','opencv-python','PyYAML','lap')},
    'cuda_available': torch.cuda.is_available(), 'cuda_runtime': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0), 'gpu_memory_bytes': torch.cuda.get_device_properties(0).total_memory,
    'device': '0', 'git_commit': subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()}
Path('outputs/video_finetune/environment.json').write_text(json.dumps(result, indent=2) + '\\n', encoding='utf-8')
print(json.dumps(result, indent=2))
"""
subprocess.run([sys.executable, '-c', environment_code], cwd=REPO, check=True)
for name in ('environment.json', 'pip_check.json', 'published_snapshot.json'):
    shutil.copy2(OUT / name, BACKUP_RUN / name)
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'scripts'))
from colab_support import digest_file
assert digest_file(REPO / 'models/best.pt') == BASELINE_SHA256
print('Verified baseline checkpoint; checkout:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())


## 4. Verify original data and the reviewed train-only addition
The original archive and every member are checked without replacing differing files. Added frames go to a separate directory. The mixed YAML lists both train directories and keeps the original valid/test paths.


In [ ]:
from colab_support import prepare_dataset_archive, extract_dataset
transfer = prepare_dataset_archive(DATA_ARCHIVE, DRIVE_PROJECT)
extraction = extract_dataset(DATA_ARCHIVE, REPO)
subprocess.run([sys.executable, 'scripts/check_dataset.py'], cwd=REPO, check=True)
(OUT / 'baseline_dataset_check.json').write_text(json.dumps(
    {'transfer': transfer, 'extraction': extraction, 'check_dataset': 'passed'}, indent=2) + '\n', encoding='utf-8')
subprocess.run([sys.executable, 'scripts/finetune_video.py', 'prepare',
    '--archive', str(DRIVE_ADDITION), '--archive-sha256', ADDITION_SHA256,
    '--addition-root', str(ADDITION_ROOT), '--data', str(MIXED_DATA)], cwd=REPO, check=True)
for name in ('baseline_dataset_check.json', 'dataset_check.json'):
    shutil.copy2(OUT / name, BACKUP_RUN / name)
print('Mixed data:', MIXED_DATA.read_text())


## 5. Fine-tune the completed model for 100 GPU epochs
Every saved epoch backs up last.pt, best.pt, CSV and settings to the new Drive run. Ultralytics internal batch changes are blocked. Only terminal CUDA OOM permits a new batch-8 or batch-4 attempt; other errors stop. Logs and failed attempts are preserved. The original validation split selects best.pt.


In [ ]:
def run_logged(command, log_path):
    print('Running:', ' '.join(map(str, command)))
    with Path(log_path).open('x', encoding='utf-8') as log:
        process = subprocess.Popen(list(map(str, command)), cwd=REPO,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            encoding='utf-8', errors='replace', bufsize=1,
            env={**os.environ, 'PYTHONUNBUFFERED': '1', 'PYTHONIOENCODING': 'utf-8'})
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        return process.wait()

if (REPO / 'models/video_finetune/best.pt').exists():
    raise FileExistsError('Completed adaptation already exists; preserve it instead of retraining.')
attempts = []
for batch in (16, 8, 4):
    log_path = SESSION / f'train_batch_{batch}.log'
    returncode = run_logged([sys.executable, '-u', 'scripts/finetune_video.py', 'train',
        '--data', MIXED_DATA, '--batch', batch, '--backup-root', BACKUP_RUN], log_path)
    attempts.append({'batch': batch, 'exit_code': returncode, 'log': str(log_path.relative_to(REPO))})
    (OUT / 'training_attempts.json').write_text(json.dumps(attempts, indent=2) + '\n', encoding='utf-8')
    shutil.copy2(OUT / 'training_attempts.json', BACKUP_RUN / 'training_attempts.json')
    if returncode == 0:
        break
    terminal_failure = log_path.read_text(encoding='utf-8').rsplit('Traceback (most recent call last):', 1)[-1]
    actual_cuda_oom = bool(re.search(r'CUDA out of memory|CUDA error: out of memory', terminal_failure, re.I))
    if not actual_cuda_oom or batch == 4:
        raise RuntimeError(f'Fine-tuning failed at batch {batch}; inspect {log_path}. No settings were silently changed.')
    print(f'Actual CUDA OOM recorded at batch {batch}; retrying only the next permitted batch.')
training = json.loads((OUT / 'training_run.json').read_text())
assert training['epochs_completed'] == 100 and training['device'].startswith('cuda')
assert digest_file(REPO / 'models/best.pt') == BASELINE_SHA256
ACTUAL_BATCH = training['actual_batch']
print(json.dumps(training, indent=2))


## 6. Evaluate the unchanged original test split
These metrics are written separately from the baseline. The original dataset's known annotation and scene-overlap limitations still apply. The source-video frames are not in this test split.


In [ ]:
status = run_logged([sys.executable, '-u', 'scripts/finetune_video.py', 'evaluate',
    '--data', MIXED_DATA, '--batch', ACTUAL_BATCH, '--backup-root', BACKUP_RUN], SESSION / 'evaluate.log')
if status:
    raise RuntimeError('Original-test evaluation failed; inspect the saved log.')
print((OUT / 'test_comparison.json').read_text())


## 7. Recheck the added training scene with both models
Both models use imgsz 640 and the same labeled video frames. For the adapted model this is **training_scene_resubstitution**, not independent validation and not a new video benchmark. It is not used to select a checkpoint.


In [ ]:
status = run_logged([sys.executable, '-u', 'scripts/finetune_video.py', 'scene-recheck',
    '--batch', ACTUAL_BATCH, '--backup-root', BACKUP_RUN], SESSION / 'training_scene_recheck.log')
if status:
    raise RuntimeError('Training-scene recheck failed; inspect the saved log.')
for kind in ('baseline', 'finetuned'):
    print((OUT / f'training_scene_recheck_{kind}.json').read_text())


## 8. Collect only the new experiment artifacts
The ZIP contains the adapted model, new measured JSON, and new train/test graphs. It does not overwrite or republish baseline files. After downloading, run the separate adapted-model video, visually inspect actual classes/IDs/trails, and label that video as a training-scene demonstration in the report.


In [ ]:
archive = Path('/content') / f'stationery_video_finetune_results_{RUN_ID}.zip'
subprocess.run([sys.executable, 'scripts/finetune_video.py', 'collect', str(archive),
                '--backup-root', str(BACKUP_RUN)], cwd=REPO, check=True)
print('Final result ZIP:', archive)
print('Drive result folder:', BACKUP_RUN)
from IPython.display import FileLink, display
display(FileLink(str(archive)))


## Runtime interruption

Preserve this run's Drive backups, logs and original dataset. Do not rerun completed training or reset/delete previous results. If the runtime disconnects, inspect the saved last.pt and matching results.csv/args.yaml before a separate resumption procedure. The paused Windows CPU process is not this run.

References: [original project](https://github.com/jeongjunseo/yolov8-stationery-tracking), [Ultralytics training](https://docs.ultralytics.com/modes/train/).
